<a href="https://colab.research.google.com/github/GinnaGomez09/proyecto_aplicado_javeriana/blob/main/notebooks/03_limpieza_normalizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ============================================================
# Clonar repositorio y preparar entorno
# ============================================================

!git clone https://github.com/GinnaGomez09/proyecto_aplicado_javeriana.git
%cd proyecto_aplicado_javeriana

Cloning into 'proyecto_aplicado_javeriana'...
remote: Enumerating objects: 339, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 339 (delta 126), reused 18 (delta 18), pack-reused 142 (from 1)
Receiving objects: 100% (339/339), 3.79 MiB | 5.37 MiB/s, done.
Resolving deltas: 100% (180/180), done.
/content/proyecto_aplicado_javeriana/proyecto_aplicado_javeriana


In [16]:
# ============================================================
# 03_LIMPIEZA_NORMALIZACION.ipynb
# Limpieza y normalización del dataset reducido de recetas
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

# ------------------------------------------------------------
# 1. Cargar dataset reducido desde carpeta interim
# ------------------------------------------------------------

recetas_path = "data/interim/recetas_reducidas.csv"

recetas = pd.read_csv(recetas_path)

print("Dimensiones del dataset reducido:", recetas.shape)
print("Número de recetas:", recetas["receta_uuid"].nunique())

recetas.head()

Dimensiones del dataset reducido: (3777, 15)
Número de recetas: 436


,receta_uuid,receta_titulo,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,cantidad_min,cantidad_max,unidad,ingrediente_nombre,tcac_alimento_codigo,tcac_alimento_nombre,match_method,match_score,cantidad_gramos_est
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,NaN,NaN,taza,harina de arepa blanca o amarilla,NaN,NaN,NaN,NaN,NaN
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1,1.000000,NaN,NaN,taza,agua tibia,NaN,NaN,NaN,NaN,NaN
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,NaN,NaN,taza,queso mozzarella o queso blanco rallado,NaN,NaN,NaN,NaN,NaN
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2,2.000000,NaN,NaN,cucharadas,mantequilla,NaN,NaN,NaN,NaN,NaN
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,NaN,NaN,NaN,NaN,NaN,Sal,NaN,NaN,NaN,NaN,NaN


In [17]:
# ------------------------------------------------------------
# 2. Conservar únicamente columnas necesarias para esta etapa
# ------------------------------------------------------------

columnas_necesarias = [
    "receta_uuid",
    "receta_titulo",
    "ingrediente_id",
    "ingrediente_linea",
    "cantidad_original",
    "cantidad_conv",
    "unidad",
    "ingrediente_nombre"
]

recetas = recetas[columnas_necesarias].copy()

print("Columnas utilizadas:")
print(recetas.columns.tolist())

Columnas utilizadas:
['receta_uuid', 'receta_titulo', 'ingrediente_id', 'ingrediente_linea', 'cantidad_original', 'cantidad_conv', 'unidad', 'ingrediente_nombre']


In [18]:
# ------------------------------------------------------------
# 3. Definir columnas clave
# ------------------------------------------------------------

col_linea = "ingrediente_linea"
col_ingrediente = "ingrediente_nombre"
col_unidad = "unidad"

print("Columna de línea de ingrediente:", col_linea)
print("Columna de ingrediente:", col_ingrediente)
print("Columna de unidad:", col_unidad)

Columna de línea de ingrediente: ingrediente_linea
Columna de ingrediente: ingrediente_nombre
Columna de unidad: unidad


In [19]:
# ------------------------------------------------------------
# 4. Funciones de limpieza textual
# ------------------------------------------------------------

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )

    return texto


def normalizar_fracciones(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    fracciones = {
        "½": "1/2",
        "⅓": "1/3",
        "⅔": "2/3",
        "¼": "1/4",
        "¾": "3/4"
    }

    for simbolo, valor in fracciones.items():
        texto = texto.replace(simbolo, valor)

    texto = re.sub(r"(\d)\s*/\s*(\d)", r"\1/\2", texto)

    return texto


def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto)

    # Normalizar fracciones antes de eliminar caracteres
    texto = normalizar_fracciones(texto)

    # Convertir a minúsculas
    texto = texto.lower()

    # Quitar tildes
    texto = quitar_tildes(texto)

    # Eliminar caracteres especiales conservando números, letras, espacios y /
    texto = re.sub(r"[^a-z0-9\s\/\.,]", " ", texto)

    # Normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [20]:
# ------------------------------------------------------------
# 5. Normalización básica de unidades
# ------------------------------------------------------------

def normalizar_unidad(texto):
    if pd.isna(texto):
        return np.nan

    texto = limpiar_texto(texto)

    equivalencias = {
        "cda": "cucharada",
        "cdas": "cucharada",
        "cucharadas": "cucharada",
        "cucharada": "cucharada",

        "cdta": "cucharadita",
        "cdtas": "cucharadita",
        "cucharaditas": "cucharadita",
        "cucharadita": "cucharadita",

        "tazas": "taza",
        "taza": "taza",
        "tz": "taza",

        "g": "gramo",
        "gr": "gramo",
        "gr.": "gramo",
        "gramos": "gramo",
        "gramo": "gramo",

        "kg": "kilogramo",
        "kg.": "kilogramo",
        "kilo": "kilogramo",
        "kilos": "kilogramo",
        "kilogramos": "kilogramo",

        "ml": "mililitro",
        "ml.": "mililitro",
        "mililitros": "mililitro",

        "l": "litro",
        "lt": "litro",
        "litros": "litro",

        "unidades": "unidad",
        "unidad": "unidad",

        "dientes": "diente",
        "diente": "diente",

        "latas": "lata",
        "lata": "lata",

        "paquetes": "paquete",
        "paquete": "paquete",

        "hojas": "hoja",
        "hoja": "hoja"
    }

    return equivalencias.get(texto, texto)

In [21]:
# ------------------------------------------------------------
# 6. Aplicar limpieza y normalización
# ------------------------------------------------------------

recetas_limpias = recetas.copy()

recetas_limpias["ingrediente_linea_limpia"] = (
    recetas_limpias[col_linea]
    .apply(limpiar_texto)
)

recetas_limpias["ingrediente_nombre_limpio"] = (
    recetas_limpias[col_ingrediente]
    .apply(limpiar_texto)
)

recetas_limpias["unidad_limpia"] = (
    recetas_limpias[col_unidad]
    .apply(normalizar_unidad)
)

recetas_limpias.head()

,receta_uuid,receta_titulo,ingrediente_id,ingrediente_linea,cantidad_original,cantidad_conv,unidad,ingrediente_nombre,ingrediente_linea_limpia,ingrediente_nombre_limpio,unidad_limpia
0,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,1,1 taza de harina de arepa blanca o amarilla,1,1.000000,taza,harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza
1,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,2,1 taza de agua tibia,1,1.000000,taza,agua tibia,1 taza de agua tibia,agua tibia,taza
2,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,3,⅓ taza de queso mozzarella o queso blanco rallado,⅓,0.333333,taza,queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,taza
3,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,4,2 cucharadas de mantequilla,2,2.000000,cucharadas,mantequilla,2 cucharadas de mantequilla,mantequilla,cucharada
4,86af61e4-e16a-11ed-9591-a96d6180cd25,Arepas de Queso,5,Sal,NaN,NaN,NaN,Sal,sal,sal,NaN


In [22]:
# ------------------------------------------------------------
# 7. Comparar antes y después
# ------------------------------------------------------------

recetas_limpias[
    [
        "ingrediente_linea",
        "ingrediente_linea_limpia",
        "ingrediente_nombre",
        "ingrediente_nombre_limpio",
        "unidad",
        "unidad_limpia"
    ]
].head(20)

,ingrediente_linea,ingrediente_linea_limpia,ingrediente_nombre,ingrediente_nombre_limpio,unidad,unidad_limpia
0,1 taza de harina de arepa blanca o amarilla,1 taza de harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,harina de arepa blanca o amarilla,taza,taza
1,1 taza de agua tibia,1 taza de agua tibia,agua tibia,agua tibia,taza,taza
2,⅓ taza de queso mozzarella o queso blanco rallado,1/3 taza de queso mozzarella o queso blanco ra...,queso mozzarella o queso blanco rallado,queso mozzarella o queso blanco rallado,taza,taza
3,2 cucharadas de mantequilla,2 cucharadas de mantequilla,mantequilla,mantequilla,cucharadas,cucharada
4,Sal,sal,Sal,sal,NaN,NaN
5,8 muslos de pollo sin la piel,8 muslos de pollo sin la piel,muslos de pollo sin la piel,muslos de pollo sin la piel,NaN,NaN
6,1 cucharada de aceite vegetal,1 cucharada de aceite vegetal,aceite vegetal,aceite vegetal,cucharada,cucharada
7,½ taza de cebolla picada,1/2 taza de cebolla picada,cebolla picada,cebolla picada,taza,taza
8,½ de taza de pimientón rojo picado,1/2 de taza de pimienton rojo picado,de taza de pimientón rojo picado,de taza de pimienton rojo picado,NaN,NaN
9,1 diente de ajo picado,1 diente de ajo picado,ajo picado,ajo picado,diente,diente


In [23]:
# ------------------------------------------------------------
# 8. Validación rápida de resultados
# ------------------------------------------------------------

print("Top 20 ingredientes limpios:")
display(
    recetas_limpias["ingrediente_nombre_limpio"]
    .value_counts()
    .head(20)
)

print("Top 20 unidades limpias:")
display(
    recetas_limpias["unidad_limpia"]
    .value_counts(dropna=False)
    .head(20)
)

Top 20 ingredientes limpios:


,count
ingrediente_nombre_limpio,
agua,109
azucar,101
comino molido,91
sal,78
sal y pimienta al gusto,69
sal y pimienta,64
mantequilla,60
ajo picados,47
extracto de vainilla,46


Top 20 unidades limpias:


,count
unidad_limpia,
NaN,1405
taza,1071
cucharada,518
cucharadita,387
libra,133
diente,115
libras,54
lata,42
rebanadas,16


In [24]:
# ------------------------------------------------------------
# 9. Revisión de muestra aleatoria
# ------------------------------------------------------------

recetas_limpias[
    [
        "ingrediente_linea",
        "ingrediente_linea_limpia",
        "ingrediente_nombre",
        "ingrediente_nombre_limpio",
        "unidad",
        "unidad_limpia"
    ]
].sample(15, random_state=42)

,ingrediente_linea,ingrediente_linea_limpia,ingrediente_nombre,ingrediente_nombre_limpio,unidad,unidad_limpia
3530,1 lata de leche condensada,1 lata de leche condensada,leche condensada,leche condensada,lata,lata
999,2 cucharadas de aceite vegetal,2 cucharadas de aceite vegetal,aceite vegetal,aceite vegetal,cucharadas,cucharada
3023,1 taza de queso mozzarella cortado en cubitos,1 taza de queso mozzarella cortado en cubitos,queso mozzarella cortado en cubitos,queso mozzarella cortado en cubitos,taza,taza
1550,2 hojas de laurel,2 hojas de laurel,laurel,laurel,hojas,hoja
2428,3 tazas de fresas frescas lavadas y cortadas p...,3 tazas de fresas frescas lavadas y cortadas p...,fresas frescas lavadas y cortadas por la mitad,fresas frescas lavadas y cortadas por la mitad,tazas,taza
3732,2 cucharadas de vinagre blanco,2 cucharadas de vinagre blanco,vinagre blanco,vinagre blanco,cucharadas,cucharada
2955,1 taza de fresas cortadas en cubitos,1 taza de fresas cortadas en cubitos,fresas cortadas en cubitos,fresas cortadas en cubitos,taza,taza
3646,1 cucharada de extracto de vainilla,1 cucharada de extracto de vainilla,extracto de vainilla,extracto de vainilla,cucharada,cucharada
465,1 cucharada de aceite,1 cucharada de aceite,aceite,aceite,cucharada,cucharada
1123,2 cucharadas de leche,2 cucharadas de leche,leche,leche,cucharadas,cucharada


In [25]:
# ------------------------------------------------------------
# 10. Guardar dataset limpio en carpeta interim
# ------------------------------------------------------------

Path("data/interim").mkdir(parents=True, exist_ok=True)

output_path = "data/interim/recetas_limpias.csv"

recetas_limpias.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset limpio guardado en:")
print(output_path)

print("\nDimensiones finales:", recetas_limpias.shape)
print("Número de recetas:", recetas_limpias["receta_uuid"].nunique())

Dataset limpio guardado en:
data/interim/recetas_limpias.csv

Dimensiones finales: (3777, 11)
Número de recetas: 436


In [26]:
recetas_limpias.to_csv("data/interim/recetas_limpias.csv", index=False)

In [27]:
from google.colab import files

files.download("data/interim/recetas_limpias.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Limpieza y normalización del dataset reducido de recetas

En esta etapa se realiza la limpieza y normalización textual del dataset reducido generado previamente en la fase de reducción y depuración de datos.

El objetivo principal es preparar las recetas para las etapas posteriores del pipeline NLP, reduciendo la variabilidad léxica y mejorando la consistencia textual de los ingredientes y unidades de medida.

Para ello, se utilizan únicamente las columnas relevantes para el procesamiento de lenguaje natural, descartando información no necesaria para esta fase.

Las transformaciones aplicadas incluyen:

- conversión de texto a minúsculas,
- eliminación de tildes y caracteres especiales,
- normalización de espacios,
- normalización de fracciones culinarias,
- limpieza de líneas de ingredientes,
- y estandarización básica de unidades de medida.

Como resultado, se generan nuevas columnas limpias y normalizadas que servirán como entrada para las siguientes etapas del proyecto:

- tokenización y lematización,
- extracción de entidades culinarias (NER),
- estandarización de ingredientes,
- conversión de unidades,
- e integración con la TCAC.

El dataset procesado se guarda en:

`data/interim/recetas_limpias.csv`

Este archivo constituye la base textual normalizada para el pipeline NLP híbrido del proyecto.